In [1]:
# ── USER CONFIG ─────────────────────────────────────────────────
# Path to your merged fine-tuned model directory
MODEL_PATH = "./llama3-housing-lora-merged"  # adjust if needed

# Path to source JSONL
RAG_JSONL = "./housing_rag_zillow.jsonl"

# ChromaDB directory
CHROMA_DIR = "./.chromadb"

TOP_K = 3
# ────────────────────────────────────────────────────────────────

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Auto-detect best device (MPS on Apple Silicon, CUDA on GPU, else CPU)
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("Model loaded successfully.")

/Users/siqili/micromamba/envs/my_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


The tokenizer you are loading from './llama3-housing-lora-merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use mps


Model loaded successfully.


In [3]:
## load dataset
from datasets import load_dataset
import json
import re
from pathlib import Path
dataset = load_dataset("json", data_files=RAG_JSONL, split="train")



SYSTEM_PROMPT = "You are a helpful housing market assistant. Answer only with information supported by the source."

def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()

def add_pair(pairs, instruction, output, source_id):
    pairs.append({
        "instruction": instruction.strip(),
        "input": "",
        "output": clean_text(output),
        "source_id": source_id
    })

def generate_pairs_from_record(record):
    text = record["text"]
    source_id = record["id"]
    pairs = []

    if source_id == "housing_zillow_001":
        add_pair(pairs, "What was the Zillow Home Value Index for Iowa City, IA as of February 28, 2026?",
                 "As of February 28, 2026, the Zillow Home Value Index for Iowa City, IA was $292,103.", source_id)
        add_pair(pairs, "What was the year-over-year change in the Iowa City Zillow Home Value Index as of February 28, 2026?",
                 "The Iowa City Zillow Home Value Index was up 4.6 percent year over year as of February 28, 2026.", source_id)
        add_pair(pairs, "What was the median sale price in Iowa City as of January 31, 2026?",
                 "The median sale price in Iowa City was $289,417 as of January 31, 2026.", source_id)
        add_pair(pairs, "What was the median list price in Iowa City in the Zillow citywide overview?",
                 "The median list price in Iowa City was $297,567.", source_id)
        add_pair(pairs, "What was the median sale-to-list ratio in Iowa City?",
                 "The median sale-to-list ratio in Iowa City was 0.983, or 98.3 percent.", source_id)
        add_pair(pairs, "How many active for-sale listings were there in Iowa City?",
                 "There were 248 active for-sale listings in Iowa City.", source_id)
        add_pair(pairs, "How many new listings were recorded in Iowa City as of February 28, 2026?",
                 "There were 54 new listings in Iowa City as of February 28, 2026.", source_id)
        add_pair(pairs, "What percent of home sales in Iowa City were over list price?",
                 "10.2 percent of home sales in Iowa City were over list price.", source_id)
        add_pair(pairs, "What percent of home sales in Iowa City were under list price?",
                 "68.1 percent of home sales in Iowa City were under list price.", source_id)
        add_pair(pairs, "What was the median number of days to pending in Iowa City?",
                 "The median days to pending in Iowa City was 56 days.", source_id)
        add_pair(pairs, "What does a 0.983 sale-to-list ratio imply for buyers in Iowa City?",
                 "It implies buyers were negotiating about 1.7 percent below asking price on average.", source_id)
        add_pair(pairs, "Which was higher in early 2026 in Iowa City, the median list price or the median sale price?",
                 "The median list price was higher. It was $297,567 compared with a median sale price of $289,417.", source_id)
        add_pair(pairs, "Were most home sales in Iowa City closing above or below list price?",
                 "Most home sales were closing below list price, since 68.1 percent of sales were under list and 10.2 percent were over list.", source_id)

    elif source_id == "housing_zillow_002":
        add_pair(pairs, "Which Iowa City neighborhood had the highest Zillow Home Value Index in February 2026?",
                 "South Pointe had the highest Zillow Home Value Index at $314,570.", source_id)
        add_pair(pairs, "Which Iowa City neighborhood had the lowest tracked Zillow Home Value Index in February 2026?",
                 "Broadway had the lowest tracked Zillow Home Value Index at $96,298.", source_id)
        add_pair(pairs, "What was the Zillow Home Value Index for Pepperwood in February 2026?",
                 "Pepperwood had a Zillow Home Value Index of $283,676 in February 2026.", source_id)
        add_pair(pairs, "What was the Zillow Home Value Index for Oak Grove in February 2026?",
                 "Oak Grove had a Zillow Home Value Index of $242,293 in February 2026.", source_id)
        add_pair(pairs, "What was the Zillow Home Value Index for Wetherby in February 2026?",
                 "Wetherby had a Zillow Home Value Index of $241,441 in February 2026.", source_id)
        add_pair(pairs, "What was the Zillow Home Value Index for Creekside in February 2026?",
                 "Creekside had a Zillow Home Value Index of $232,169 in February 2026.", source_id)
        add_pair(pairs, "What was the Zillow Home Value Index for Lucas Farms in February 2026?",
                 "Lucas Farms had a Zillow Home Value Index of $223,431 in February 2026.", source_id)
        add_pair(pairs, "What was the Zillow Home Value Index for Grant Wood in February 2026?",
                 "Grant Wood had a Zillow Home Value Index of $218,543 in February 2026.", source_id)
        add_pair(pairs, "Did the neighborhood breakdown report enough data for Paddock?",
                 "No. Paddock was listed as not available because there was insufficient data.", source_id)
        add_pair(pairs, "Approximately how much more expensive was South Pointe than Broadway according to the neighborhood breakdown?",
                 "South Pointe commanded roughly a 44 percent premium over Broadway.", source_id)
        add_pair(pairs, "What explanation does the source give for Broadway's low home value index?",
                 "The source suggests Broadway's low value likely reflects a high share of multi-family and student housing.", source_id)
        add_pair(pairs, "Which neighborhoods were described as clustering in the $218,000 to $242,000 range?",
                 "Grant Wood, Wetherby, and Creekside were described as clustering in the $218,000 to $242,000 range.", source_id)

    elif source_id == "housing_zillow_003":
        add_pair(pairs, "What was the average rent in Iowa City as of February 28, 2026?",
                 "The average rent in Iowa City was $1,308 per month as of February 28, 2026.", source_id)
        add_pair(pairs, "What was the national average rent in the Zillow rent comparison?",
                 "The national average rent in the comparison was $1,895 per month.", source_id)
        add_pair(pairs, "What was the month-over-month rent change in Iowa City?",
                 "The month-over-month rent change in Iowa City was plus 0.4 percent.", source_id)
        add_pair(pairs, "What was the year-over-year rent change in Iowa City?",
                 "The year-over-year rent change in Iowa City was plus 4.3 percent.", source_id)
        add_pair(pairs, "How did Iowa City average rent compare with the national average?",
                 "Iowa City average rent was approximately 31 percent below the national average.", source_id)
        add_pair(pairs, "Why does the source say Iowa City rents are below the national average?",
                 "The source attributes lower rents to Iowa's generally lower cost of living and the dominance of student rental housing, which increases supply near the University of Iowa campus.", source_id)
        add_pair(pairs, "What does the source suggest about the 4.3 percent year-over-year rent increase?",
                 "The source suggests the increase reflects continued demand pressure in the rental market despite the availability of affordable for-sale options.", source_id)
        add_pair(pairs, "Why does the source say the Zillow Observed Rent Index is a cleaner measure than raw asking-price averages?",
                 "The source says ZORI controls for changes in the quality of available rental inventory, making it a cleaner measure of rent-level trends than raw asking-price averages.", source_id)
        add_pair(pairs, "How much lower was Iowa City average rent than the national average in dollar terms?",
                 "Iowa City average rent was $587 lower per month than the national average.", source_id)

    return pairs

def flat_to_messages(example):
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["output"]},
        ],
        "source_id": example["source_id"]
    }

def main(input_path, flat_output_path, messages_output_path):
    records = []
    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    flat_examples = []
    for record in records:
        flat_examples.extend(generate_pairs_from_record(record))

    message_examples = [flat_to_messages(ex) for ex in flat_examples]

    Path(flat_output_path).parent.mkdir(parents=True, exist_ok=True)

    with open(flat_output_path, "w", encoding="utf-8") as f:
        for ex in flat_examples:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

    with open(messages_output_path, "w", encoding="utf-8") as f:
        for ex in message_examples:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

    print(f"Wrote {len(flat_examples)} flat QA examples to {flat_output_path}")
    print(f"Wrote {len(message_examples)} message-format examples to {messages_output_path}")

if __name__ == "__main__":
    main(
        input_path="housing_source.jsonl",
        flat_output_path="finetune_qa.jsonl",
        messages_output_path="finetune_messages.jsonl"
    )

Generating train split: 3 examples [00:00, 355.03 examples/s]

Wrote 34 flat QA examples to finetune_qa.jsonl
Wrote 34 message-format examples to finetune_messages.jsonl


In [4]:


from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig 
from transformers import TrainingArguments
import torch

hgdataset = load_dataset("json", data_files="/Users/siqili/Documents/GitHub/MarketMindAI/finetune_messages.jsonl", split="train")
MODEL_ID = "meta-llama/Llama-3.2-1B-instruct"

# 4-bit QLoRA quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# LoRA config — targets attention projection layers
lora_config = LoraConfig(
    r=16,                     # rank
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # ~0.04% of total params

training_args = SFTConfig(
    output_dir="./llama3-lora-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_length=2048,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hgdataset,  # HF dataset with "text" column
    args=training_args,
)
trainer.train()

Generating train split: 34 examples [00:00, 34454.30 examples/s]


trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


Truncating train dataset: 100%|██████████| 34/34 [00:00<00:00, 4412.73 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.
/Users/siqili/micromamba/envs/my_env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


TrainOutput(global_step=9, training_loss=3.3152103424072266, metrics={'train_runtime': 113.5898, 'train_samples_per_second': 0.898, 'train_steps_per_second': 0.079, 'total_flos': 57948713410560.0, 'train_loss': 3.3152103424072266})

In [5]:
from peft import PeftModel

# Merge LoRA weights into base model
merged_model = model.merge_and_unload()
merged_model.save_pretrained("llama3-housing-lora-merged")
tokenizer.save_pretrained("llama3-housing-lora-merged")

# Push to HF Hub
merged_model.push_to_hub("SiwgiLi/llama3-housing-lora-finetuned")
tokenizer.push_to_hub("SiwgiLi/llama3-housing-lora-finetuned")

/Users/siqili/micromamba/envs/my_env/lib/python3.11/site-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
Processing Files (1 / 1): 100%|██████████| 1.07GB / 1.07GB,  376MB/s  
New Data Upload: 100%|██████████| 12.7MB / 12.7MB, 5.30MB/s  
Processing Files (1 / 1): 100%|██████████| 17.2MB / 17.2MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/SiwgiLi/llama3-housing-lora-finetuned/commit/0cb97b6e4023916a7a193ad953de6a621287ecf6', commit_message='Upload tokenizer', commit_description='', oid='0cb97b6e4023916a7a193ad953de6a621287ecf6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/SiwgiLi/llama3-housing-lora-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='SiwgiLi/llama3-housing-lora-finetuned'), pr_revision=None, pr_num=None)

In [6]:
import json
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 1. Load your source JSONL
documents = []

with open("housing_rag_zillow.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        documents.append(
            Document(
                page_content=record["text"],
                metadata={
                    "id": record["id"],
                    "source_name": record["source_name"],
                    "source_url": record["source_url"],
                    "publication_date": record["publication_date"],
                    "section": record["section"],
                    "topic_tags": ", ".join(record["topic_tags"]),
                    "word_count": record["word_count"],
                },
            )
        )

# 2. Set up embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

# 3. Create persistent Chroma vector store
#vectorstore = Chroma.from_documents(
#    documents=documents,
#    embedding=embeddings,
#    collection_name="iowa_housing_rag",
#    persist_directory="./chroma_db"
#)

# 4. Create retriever
#retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

vectorstore = Chroma(
    collection_name="iowa_housing_rag",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [8]:
SYSTEM_PROMPT = (
    "You are a helpful housing market assistant. "
    "Answer only with information supported by the source."
)

def rag_query(user_question: str) -> str:
    docs = retriever.invoke(user_question)

    context_blocks = []
    for doc in docs:
        block = (
            f"[Source: {doc.metadata.get('source_name', 'Unknown')} | "
            f"Section: {doc.metadata.get('section', 'Unknown')} | "
            f"Date: {doc.metadata.get('publication_date', 'Unknown')}]\n"
            f"{doc.page_content}"
        )
        context_blocks.append(block)

    context = "\n\n".join(context_blocks)

    # Llama 3 instruct chat format
    prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n"
        f"{SYSTEM_PROMPT}\n<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n"
        f"Context:\n{context}\n\nQuestion: {user_question}\n<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
    )

    output = pipe(
        prompt,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,  # avoids padding warning
    )

    # Strip the prompt, return only the new generated text
    full_text = output[0]["generated_text"]
    answer = full_text.split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()
    return answer

In [9]:
results = vectorstore.similarity_search(
    query="What is the average rent in Iowa City?",
    k=3,
    filter={"section": "Zillow Observed Rent Index (ZORI) — Iowa City"}
)

for doc in results:
    print(doc.metadata["id"], doc.page_content[:150])

housing_zillow_003 Iowa City Rental Market (Zillow Observed Rent Index / ZORI, as of February 28, 2026):

 Average Rent (Iowa City): $1,308/month
 National Average Rent:


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# Load your saved fine-tuned model
model_path = "./output"  # or wherever SFTTrainer saved it

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"  # uses MPS on your Mac
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

def query_rag(user_question, context):
    prompt = f"""<|system|>
You are a helpful housing market assistant. Answer only with information supported by the source.
<|user|>
Context: {context}

Question: {user_question}
<|assistant|>"""
    output = pipe(prompt, max_new_tokens=256, do_sample=False)
    return output[0]["generated_text"].split("<|assistant|>")[-1].strip()

# Test it
print(query_rag("What is the median sale price in Iowa City?", your_retrieved_context))

HFValidationError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: './output'.